In [ ]:
# Cell 1: Mount Drive and Install Packages
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/PhishGuard'

import subprocess
subprocess.run([
    'pip', 'install',
    'xgboost==2.1.0',
    'shap==0.45.0',
    'scikit-learn==1.5.0',
    'pandas==2.2.0',
    'numpy==1.26.0',
    'joblib==1.4.0',
    'matplotlib',
    'seaborn',
    '-q'
], check=False)

import os

MODEL_FILES = [
    'wallet_xgboost_v1.pkl',
    'wallet_feature_schema.json',
    'wallet_test_indices.npy',
    'contract_xgboost_v1.pkl',
    'contract_feature_schema.json',
    'contract_test_indices.npy',
    'contract_threshold.json',
]

FEATURE_FILES = [
    'wallet_features.csv',
    'contract_features.csv',
]

for fname in MODEL_FILES:
    path = f'{DRIVE_BASE}/models/{fname}'
    assert os.path.exists(path), f'Missing model file: {path}'
    print(f'OK  models/{fname}')

for fname in FEATURE_FILES:
    path = f'{DRIVE_BASE}/features/{fname}'
    assert os.path.exists(path), f'Missing feature file: {path}'
    print(f'OK  features/{fname}')

print('\nCell 1 ready.')


In [ ]:
# Cell 2: Load models, features and test sets 
import os
import json
import numpy as np
import pandas as pd
import joblib
import xgboost as xgb
import shap
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve
)

# ── Wallet ────────────────────────────────────────────────────
with open(f'{DRIVE_BASE}/models/wallet_feature_schema.json') as f:
    wallet_feature_cols = json.load(f)

df_wallet = pd.read_csv(f'{DRIVE_BASE}/features/wallet_features.csv')
X_wallet  = df_wallet[wallet_feature_cols].values
y_wallet  = df_wallet['label'].values

wallet_test_indices = np.load(f'{DRIVE_BASE}/models/wallet_test_indices.npy')
X_wallet_test = X_wallet[wallet_test_indices]
y_wallet_test = y_wallet[wallet_test_indices]

wallet_model = joblib.load(f'{DRIVE_BASE}/models/wallet_xgboost_v1.pkl')

# ── Contract ──────────────────────────────────────────────────
with open(f'{DRIVE_BASE}/models/contract_feature_schema.json') as f:
    contract_feature_cols = json.load(f)

df_contract = pd.read_csv(f'{DRIVE_BASE}/features/contract_features.csv')
X_contract  = df_contract[contract_feature_cols].values
y_contract  = df_contract['label'].values

contract_test_indices = np.load(f'{DRIVE_BASE}/models/contract_test_indices.npy')
X_contract_test = X_contract[contract_test_indices]
y_contract_test = y_contract[contract_test_indices]

contract_model = joblib.load(f'{DRIVE_BASE}/models/contract_xgboost_v1.pkl')

with open(f'{DRIVE_BASE}/models/contract_threshold.json') as f:
    threshold_data = json.load(f)
contract_threshold = threshold_data['threshold']

# ── Thresholds ────────────────────────────────────────────────
wallet_threshold = 0.5  # default — no tuning needed given 99.38% accuracy

# ── Shapes ────────────────────────────────────────────────────
print(f'Wallet   features : {X_wallet.shape}   |  test set: {X_wallet_test.shape}')
print(f'Contract features : {X_contract.shape} |  test set: {X_contract_test.shape}')
print(f'Wallet   threshold: {wallet_threshold}')
print(f'Contract threshold: {contract_threshold}')

assert X_wallet_test.shape[0]  == 800, \
    f'Expected 800 wallet test rows, got {X_wallet_test.shape[0]}'
assert X_contract_test.shape[0] == 390, \
    f'Expected 390 contract test rows, got {X_contract_test.shape[0]}'

print('\nCell 2 ready.')


In [ ]:
# Cell 3: COmpute All Metrics for Both Models
# ── Probabilities ─────────────────────────────────────────────
wallet_proba   = wallet_model.predict_proba(X_wallet_test)[:, 1]
contract_proba = contract_model.predict_proba(X_contract_test)[:, 1]

# ── Predictions at threshold ──────────────────────────────────
wallet_preds   = (wallet_proba   >= wallet_threshold).astype(int)
contract_preds = (contract_proba >= contract_threshold).astype(int)

# ── Wallet metrics ────────────────────────────────────────────
wallet_acc     = accuracy_score(y_wallet_test,   wallet_preds)
wallet_prec    = precision_score(y_wallet_test,  wallet_preds)
wallet_rec     = recall_score(y_wallet_test,     wallet_preds)
wallet_f1      = f1_score(y_wallet_test,         wallet_preds)
wallet_roc_auc = roc_auc_score(y_wallet_test,    wallet_proba)
wallet_pr_auc  = average_precision_score(y_wallet_test, wallet_proba)
wallet_cm      = confusion_matrix(y_wallet_test, wallet_preds)

# ── Contract metrics ──────────────────────────────────────────
contract_acc     = accuracy_score(y_contract_test,   contract_preds)
contract_prec    = precision_score(y_contract_test,  contract_preds)
contract_rec     = recall_score(y_contract_test,     contract_preds)
contract_f1      = f1_score(y_contract_test,         contract_preds)
contract_roc_auc = roc_auc_score(y_contract_test,    contract_proba)
contract_pr_auc  = average_precision_score(y_contract_test, contract_proba)
contract_cm      = confusion_matrix(y_contract_test, contract_preds)

# ── Classification reports ────────────────────────────────────
print('=' * 60)
print('WALLET MODEL — Classification Report')
print(f'Threshold: {wallet_threshold}')
print('=' * 60)
print(classification_report(y_wallet_test, wallet_preds,
                             target_names=['Benign', 'Phishing'],
                             digits=6))

print('=' * 60)
print('CONTRACT MODEL — Classification Report')
print(f'Threshold: {contract_threshold:.4f}')
print('=' * 60)
print(classification_report(y_contract_test, contract_preds,
                             target_names=['Benign', 'Phishing'],
                             digits=6))

# ── Confusion matrices ────────────────────────────────────────
def print_cm(cm, model_name):
    tn, fp, fn, tp = cm.ravel()
    print(f'\n{model_name} — Confusion Matrix')
    print(f'  {"":12s}  Pred Benign  Pred Phishing')
    print(f'  {"True Benign":12s}  TN={tn:<10} FP={fp}')
    print(f'  {"True Phishing":12s}  FN={fn:<10} TP={tp}')

print_cm(wallet_cm,   'WALLET MODEL')
print_cm(contract_cm, 'CONTRACT MODEL')

# ── Summary table ─────────────────────────────────────────────
print('\n' + '=' * 60)
print(f'{"Accuracy":<25} {wallet_acc:>12.6f} {contract_acc:>12.6f}')
print(f'{"Precision":<25} {wallet_prec:>12.6f} {contract_prec:>12.6f}')
print(f'{"Recall":<25} {wallet_rec:>12.6f} {contract_rec:>12.6f}')
print(f'{"F1 Score":<25} {wallet_f1:>12.6f} {contract_f1:>12.6f}')
print(f'{"ROC-AUC":<25} {wallet_roc_auc:>12.6f} {contract_roc_auc:>12.6f}')
print(f'{"PR-AUC":<25} {wallet_pr_auc:>12.6f} {contract_pr_auc:>12.6f}')
print('=' * 60)

print('\nCell 3 ready.')


In [ ]:
# Cell 4: Generate Evaluation PLots 
EVAL_DIR = f'{DRIVE_BASE}/evaluation'
os.makedirs(EVAL_DIR, exist_ok=True)

sns.set_style('whitegrid')
WALLET_COLOR   = '#1f77b4'  # blue
CONTRACT_COLOR = '#ff7f0e'  # orange

def save_fig(fig, filename):
    path = f'{EVAL_DIR}/{filename}'
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved: {path}')

# ── Plot 1: ROC Curves ────────────────────────────────────────
fpr_w, tpr_w, _ = roc_curve(y_wallet_test,   wallet_proba)
fpr_c, tpr_c, _ = roc_curve(y_contract_test, contract_proba)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(fpr_w, tpr_w, color=WALLET_COLOR,
        label=f'Wallet   (AUC = {wallet_roc_auc:.4f})', linewidth=2)
ax.plot(fpr_c, tpr_c, color=CONTRACT_COLOR,
        label=f'Contract (AUC = {contract_roc_auc:.4f})', linewidth=2)
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
ax.set_xlabel('False Positive Rate', fontsize=13)
ax.set_ylabel('True Positive Rate', fontsize=13)
ax.set_title('ROC Curve — Wallet vs Contract Model', fontsize=15)
ax.legend(fontsize=12)
save_fig(fig, 'roc_curve.png')

# ── Plot 2: PR Curves ─────────────────────────────────────────
prec_w, rec_w, _ = precision_recall_curve(y_wallet_test,   wallet_proba)
prec_c, rec_c, _ = precision_recall_curve(y_contract_test, contract_proba)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(rec_w, prec_w, color=WALLET_COLOR,
        label=f'Wallet   (PR-AUC = {wallet_pr_auc:.4f})', linewidth=2)
ax.plot(rec_c, prec_c, color=CONTRACT_COLOR,
        label=f'Contract (PR-AUC = {contract_pr_auc:.4f})', linewidth=2)
ax.set_xlabel('Recall', fontsize=13)
ax.set_ylabel('Precision', fontsize=13)
ax.set_title('Precision-Recall Curve — Wallet vs Contract Model', fontsize=15)
ax.legend(fontsize=12)
save_fig(fig, 'pr_curve.png')

# ── Plot 3: Wallet Confusion Matrix ───────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(wallet_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred Benign', 'Pred Phishing'],
            yticklabels=['True Benign', 'True Phishing'],
            annot_kws={'size': 16}, ax=ax)
ax.set_title('Confusion Matrix — Wallet Model', fontsize=15)
ax.set_ylabel('Actual', fontsize=13)
ax.set_xlabel('Predicted', fontsize=13)
save_fig(fig, 'confusion_matrix_wallet.png')

# ── Plot 4: Contract Confusion Matrix ─────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(contract_cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Pred Benign', 'Pred Phishing'],
            yticklabels=['True Benign', 'True Phishing'],
            annot_kws={'size': 16}, ax=ax)
ax.set_title('Confusion Matrix — Contract Model', fontsize=15)
ax.set_ylabel('Actual', fontsize=13)
ax.set_xlabel('Predicted', fontsize=13)
save_fig(fig, 'confusion_matrix_contract.png')

# ── Plot 5: SHAP Beeswarm — Wallet ────────────────────────────
wallet_explainer  = shap.TreeExplainer(wallet_model)
wallet_shap_vals  = wallet_explainer.shap_values(X_wallet_test)

fig = plt.figure(figsize=(12, 8))
shap.summary_plot(
    wallet_shap_vals, X_wallet_test,
    feature_names=wallet_feature_cols,
    plot_type='dot', show=False
)
plt.title('SHAP Beeswarm — Wallet Model', fontsize=15, pad=12)
save_fig(fig, 'shap_beeswarm_wallet.png')

# ── Plot 6: SHAP Beeswarm — Contract ─────────────────────────
contract_explainer = shap.TreeExplainer(contract_model)
contract_shap_vals = contract_explainer.shap_values(X_contract_test)

fig = plt.figure(figsize=(12, 8))
shap.summary_plot(
    contract_shap_vals, X_contract_test,
    feature_names=contract_feature_cols,
    plot_type='dot', show=False
)
plt.title('SHAP Beeswarm — Contract Model', fontsize=15, pad=12)
save_fig(fig, 'shap_beeswarm_contract.png')

print('\nCell 4 ready.')


In [ ]:
# Cell 5: Feature Importance Comparision
# ── Mean absolute SHAP values ─────────────────────────────────
wallet_mean_shap   = np.abs(wallet_shap_vals).mean(axis=0)
contract_mean_shap = np.abs(contract_shap_vals).mean(axis=0)

wallet_importance_df = pd.DataFrame({
    'feature':    wallet_feature_cols,
    'mean_shap':  wallet_mean_shap
}).sort_values('mean_shap', ascending=False).reset_index(drop=True)

contract_importance_df = pd.DataFrame({
    'feature':    contract_feature_cols,
    'mean_shap':  contract_mean_shap
}).sort_values('mean_shap', ascending=False).reset_index(drop=True)

# ── Print ranked tables ───────────────────────────────────────
print('=' * 55)
print('WALLET MODEL — Feature Importance (all 23 features)')
print('=' * 55)
print(f'  {"Rank":<6} {"Feature":<45} {"Mean |SHAP|":>11}')
print('-' * 55)
for i, row in wallet_importance_df.iterrows():
    print(f'  {i+1:<6} {row["feature"]:<45} {row["mean_shap"]:>11.5f}')

print('\n' + '=' * 55)
print('CONTRACT MODEL — Feature Importance (all 21 features)')
print('=' * 55)
print(f'  {"Rank":<6} {"Feature":<45} {"Mean |SHAP|":>11}')
print('-' * 55)
for i, row in contract_importance_df.iterrows():
    print(f'  {i+1:<6} {row["feature"]:<45} {row["mean_shap"]:>11.5f}')

# ── Side-by-side bar chart: top 10 ───────────────────────────
top_w = wallet_importance_df.head(10)
top_c = contract_importance_df.head(10)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

ax1.barh(top_w['feature'][::-1], top_w['mean_shap'][::-1],
         color=WALLET_COLOR, edgecolor='white')
ax1.set_xlabel('Mean |SHAP Value|', fontsize=12)
ax1.set_title('Wallet Model — Top 10 Features', fontsize=14)
ax1.tick_params(axis='y', labelsize=11)

ax2.barh(top_c['feature'][::-1], top_c['mean_shap'][::-1],
         color=CONTRACT_COLOR, edgecolor='white')
ax2.set_xlabel('Mean |SHAP Value|', fontsize=12)
ax2.set_title('Contract Model — Top 10 Features', fontsize=14)
ax2.tick_params(axis='y', labelsize=11)

fig.suptitle('Feature Importance Comparison — Top 10 by Mean |SHAP|',
             fontsize=15, y=1.02)
plt.tight_layout()

path = f'{EVAL_DIR}/feature_importance_comparison.png'
fig.savefig(path, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'\nSaved: {path}')

print('\nCell 5 ready.')
